In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

model = init_chat_model(model="gpt-5-nano")

In [3]:
response = model.invoke("What's the capital of Moon?")
response.content

'There isn’t one. The Moon isn’t a country and has no government, so it has no capital city.\n\nIf you were thinking of a fictional universe or a place named Moon (like Moon Township, PA), tell me which one and I can help with that.'

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
agent = create_agent(model=model)

question = HumanMessage(content="What the capitol of Moon?")

response = agent.invoke(
    {"messages":[question]}
)
print(response['messages'][1].content)


There isn’t a capital or capitol for the Moon. The Moon isn’t a country and has no sovereign government, so it has no capital city. Any bases or settlements on the Moon would be governed by whoever operates them (space agencies, companies, or international agreements), not a national capital.

If you meant a fictional Moon from a book, movie, or game, tell me the title and I can tell you what that work calls its capital (if anything).


In [8]:
system_prompt= "you are a science fiction writer, create a capital city as user's request."

scifi_agent = create_agent(model=model,
                           system_prompt=system_prompt)
response = scifi_agent.invoke(
    {
        "messages":[question]
    }
)
print(response['messages'][1].content)

In this sci‑fi setting, the Moon’s capital is Lunopolis, the City of First Light.

Key highlights:
- Location: built around the rim of Shackleton Crater in the Moon’s south polar region, with habitats tucked into lava-tube corridors and protected by glass domes that harvest sunlight.
- Government: a federal capital governed by the Solar Assembly, with an Overseer Council meeting in the central Helios Dome.
- Architecture: a blend of pale regolith stone, translucent polycrystal domes, and gravity-tuned towers; transit runs on mag-lev rails and through orbital and lunar elevator links.
- Districts: Crater Rim Precinct (administrative core), Glass Crescent (cultural and civic life), Ice Cathedral (water ice reservoir and spiritual center), Lantern Quarter (residential and markets).
- Economy: powered by solar energy and helium-3 mining, with ice harvesting, photon-lattice manufacturing, and a bustling lunar-port for trade with orbit and Earth.
- Landmarks: The Crown (a towering solar-coll

In [9]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

Lunopolis


In [11]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

Name: Lunaris Prime
Location: Inside Shackleton Crater, Moon's south pole
Vibe: Frosted neon metropolis
Economy: Helium-3 mining; water ice extraction; solar power generation; microgravity manufacturing


In [15]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(model=model,
                     system_prompt="You are a science fiction writer, create a space capital city at the users request.",
                     response_format=CapitalInfo)
question = HumanMessage(content="What is the capital of Moon?")

response = agent.invoke(
    {
        "messages":[question]
    }
)
#print(response['structured_response'] )
print(f"{response['structured_response'].name} is a city located at {response['structured_response'].location}")


Selene Prime is a city located at Southern polar rim of Shackleton Crater, perched on elevated terraces above permanently shadowed basins, ringed by solar towers and orbital transit corridors.


In [5]:
#tools
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str)->Dict[str,Any]:
    """Search the web for information"""
    return tavily_client.search(query)

search_agent = create_agent(model=model,
                            tools=[web_search])

response = search_agent.invoke(
    {"messages":HumanMessage(content="Who is the current mayor of Delhi?")}
)
response


{'messages': [HumanMessage(content='Who is the current mayor of Delhi?', additional_kwargs={}, response_metadata={}, id='f55d98d8-f179-4121-a72e-f0faae8bec24'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 413, 'prompt_tokens': 132, 'total_tokens': 545, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DQ2ciDSB3w8ER1zSYMy3KSoY6BMKq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d4c24-f4a9-7f00-8dd5-5a0d5b3792ae-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of Delhi 2026'}, 'id': 'call_pUBsbr29JtcTS3AizIC2cCE1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [1]:
#personal chef


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

tavily_client = TavilyClient()

@tool
def web_search(query: str)-> Dict[str, Any]:
    """search the web for information"""
    return tavily_client.search(query)

system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

config = {'configurable':{"thread_id":"1"}}

response = agent.invoke(
    {"messages":[HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config 
)
print(response['messages'][-1].content)

Nice! Leftover chicken and rice are a great base for quick, tasty meals. Here are some solid options you can make with those ingredients (and a few common pantry add-ins). I’ve linked reputable recipes you can open for full instructions.

- Leftover Chicken Egg Fried Rice
  - Why: Uses both leftovers, quick 15–20 minutes, very adaptable with whatever veggies you have.
  - What you’ll likely need: eggs, soy sauce or tamari, onion, peas or other veggies.
  - Link: Easy Peasy Foodie — Leftover Chicken and Egg Fried Rice

- Classic Chicken Fried Rice (one-pan)
  - Why: A comforting, fast fried rice with chicken, eggs, and veggies. Great with day-old rice.
  - What you’ll need: chicken, rice, eggs, peas/carrots, green onions, soy sauce.
  - Link: Cooking Classy — Chicken Fried Rice (also see other chicken fried rice variants)

- One-Pot Chicken and Rice (creamy, comforting)
  - Why: All-in-one skillet. Simple weeknight dinner that uses pantry staples.
  - What you’ll need: chicken (breasts 

MCP

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)
tools = await client.get_tools()

agent = create_agent(
    model = model,
    tools= tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

human_message = {"messages":[HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]}

config = {"configurable":{"thread_id":1}}

response = await agent.ainvoke(
    human_message,
    config
)
print(response["messages"][-1].content)


Direct non-stop flights from San Francisco (SFO) to Tokyo (NRT or HND) on March 31 are not available. Here are the closest options within +/-3 days, all with a layover in Taipei (TPE):

Cheapest options
| Route (including layovers) | Times (Local) | Cabin | Price | Deep link |
|---|---|---|---|---|
| San Francisco SFO → Taipei TPE → Tokyo NRT | 28/03 00:10 → 29/03 19:40 (27h 30m) | Economy | 601 USD | https://on.kiwi.com/MVbHVh |
| San Francisco SFO → Taipei TPE → Tokyo NRT | 28/03 00:40 → 29/03 12:25 (27h 00m) | Economy | 601 USD | https://on.kiwi.com/ErApUU |
| San Francisco SFO → Taipei TPE → Tokyo NRT | 28/03 00:40 → 29/03 12:25 (27h 00m) | Economy | 601 USD | https://on.kiwi.com/J2t5bB |

Shortest-duration options
| Route (including layovers) | Times (Local) | Cabin | Duration | Price | Deep link |
|---|---|---|---|---|---|
| San Francisco SFO → Taipei TPE → Tokyo NRT | 28/03 00:40 → 29/03 13:25 (27h 00m) | Economy | 27h 0m | 772 USD | https://on.kiwi.com/yRPYDt |

Other interesti

CONTEXT

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool , ToolRuntime
from pprint import pprint

@dataclass
class ColorContext:
    favourite_color: str ="blue"
    least_favourite_color: str ="yellow"

@tool
def get_favourite_color(runtime: ToolRuntime)->str:
    """Gets the favourite color of user"""
    return runtime.context.favourite_color
@tool
def least_favourite_color(runtime: ToolRuntime)->str:
    """Gets the favourite color of user"""
    return runtime.context.least_favourite_color

agent = create_agent(
    model=model,
    tools=[get_favourite_color,least_favourite_color],
    context_schema=ColorContext
)

response = agent.invoke(
    {"messages":[HumanMessage(content="What is my least favourite color?")]},
    context=ColorContext()
)

pprint(response)

c:\Users\deepa\anaconda3\envs\agents\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my least favourite color?', additional_kwargs={}, response_metadata={}, id='804438fb-60f3-4d59-a3e7-39ebd78e734b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 145, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DRKiQcUDjlJ76Ljl9OARiuO73qnxd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5e7e-be22-7662-8a20-bd028994ff1f-0', tool_calls=[{'name': 'least_favourite_color', 'args': {}, 'id': 'call_wI2u43mRQmDdotMRg7kJFahq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'o

In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext(favourite_color="green")
)

c:\Users\deepa\anaconda3\envs\agents\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favourite_co...avourite_color='yellow'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


In [13]:
response

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='a2aac8f0-4f83-4964-8269-de7916871552'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 144, 'total_tokens': 357, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DRKnZUGRpnWDlEsliq47d9VSwKfNA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d5e83-9b97-78b0-938e-085f2cc05ff4-0', tool_calls=[{'name': 'get_favourite_color', 'args': {}, 'id': 'call_H0KTm3R8abQWVqSOCZ6DtPGt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_tokens': 213,

STATE


In [ ]:
from langchain.agents import AgentState,create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command 
from langchain.messages import ToolMessage, HumanMessage
from pprint import pprint

class CustomState(AgentState):
    favourite_color : str

@tool
def update_favourite_color(favourite_color:str,runtime: ToolRuntime)->Command:
    """Update the favourite color of the user once they have revealed it."""
    return Command(update={
        "favourite_color": favourite_color,
        "messages":[ToolMessage("Successfully updated favourite color",tool_call_id=runtime.tool_call_id)]
    })

agent = create_agent(
    model=model,
    tools=[update_favourite_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

response = agent.invoke(
    {"messages":[HumanMessage(content="My favourite color is green")]},
    {"configurable":{"thread_id":"1"}}
)

response = agent.invoke(
    {
        "messages":[HumanMessage(content="Hello, how are you?")],
        "favourite_color":"green"
    },
    {"configurable":{"thread_id":"10"}}
)

@tool
def read_favourite_color(runtime:ToolRuntime)->str:
    """Read the favourite color of the user from the state"""
    try:
        return runtime.state["favourite_color"]
    except KeyError:
        return "No favorite colour found in state"
    

response = agent.invoke(
    {"messages":[HumanMessage(content="What's my favourite color")]},
    {"configurable":{"thread_id":"1"}}
)
pprint(response)

{'favourite_color': 'green',
 'messages': [HumanMessage(content='My favourite color is green', additional_kwargs={}, response_metadata={}, id='0e394097-c65f-4dff-9716-09758d28c55e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 139, 'total_tokens': 422, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DRUNfMZ3cZjYTRZyYQkcqkjFhVtT7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d60b5-a7de-72b2-a732-021a7da361ce-0', tool_calls=[{'name': 'update_favourite_color', 'args': {'favourite_color': 'green'}, 'id': 'call_M6gKrn8fzKvob7PihA4Umudz', 'type': 'tool_call'}], invalid_tool

Creating Subagents

In [4]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pprint import pprint

@tool
def square_root(x: float)->float:
    """Calculate the square root of a number"""
    return x**0.5

@tool
def square(x:float)->float:
    """Calculate the square of a number"""
    return x**2

subagent_1 = create_agent(
    model=model,
    tools =[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages":[HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 1 in order to calculate the square of a number"""
    response = subagent_1.invoke({"messages":[HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

In [6]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

pprint(response["messages"][-1].content)

'The square root of 456 is approximately 21.354156504062622, or about 21.35416.'
